# Taller #37: Reconocimiento de Acciones Simples con Detección de Postura
#### Desarrollado por: David Santiago Cruz Hernández

In [1]:
import cv2
import mediapipe as mp
import numpy as np

# pip install -r requirements.txt

### Inicializar MediaPipe Pose

In [ ]:
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

### Definir Puntos Clave

In [ ]:
indices_puntos = {
    'nose': 0,
    'left_shoulder': 11,
    'right_shoulder': 12,
    'left_elbow': 13,
    'right_elbow': 14,
    'left_wrist': 15,
    'right_wrist': 16,
    'left_hip': 23,
    'right_hip': 24,
    'left_knee': 25,
    'right_knee': 26,
    'left_ankle': 27,
    'right_ankle': 28
}

# Umbral para detectar movimiento (para caminar)
umbral_movimiento = 5.0

# Variables para seguimiento de movimiento
posicion_previa_tobillos = None

### Calcular Distancia entre Puntos

In [ ]:
def calcular_distancia(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

### Obtener Coordenadas de Puntos Clave

In [ ]:
def obtener_poses():
    left_shoulder = get_coords(mp_pose.PoseLandmark.LEFT_SHOULDER.value)
    right_shoulder = get_coords(mp_pose.PoseLandmark.RIGHT_SHOULDER.value)
    left_wrist = get_coords(mp_pose.PoseLandmark.LEFT_WRIST.value)
    right_wrist = get_coords(mp_pose.PoseLandmark.RIGHT_WRIST.value)
    left_hip = get_coords(mp_pose.PoseLandmark.LEFT_HIP.value)
    right_hip = get_coords(mp_pose.PoseLandmark.RIGHT_HIP.value)
    left_knee = get_coords(mp_pose.PoseLandmark.LEFT_KNEE.value)
    right_knee = get_coords(mp_pose.PoseLandmark.RIGHT_KNEE.value)
    left_ankle = get_coords(mp_pose.PoseLandmark.LEFT_ANKLE.value)
    right_ankle = get_coords(mp_pose.PoseLandmark.RIGHT_ANKLE.value)

    return left_shoulder, right_shoulder, left_wrist, right_wrist, left_hip, right_hip, left_knee, right_knee, left_ankle, right_ankle


### Captura de Video y Reconocimiento de Acciones

In [ ]:
# Captura de video desde webcam
cap = cv2.VideoCapture(1)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Convertir a RGB
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)

    action = "ninguna"

    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark
        h, w, _ = frame.shape

        # Obtener coordenadas de puntos clave
        try:
            # Coordenadas en pixeles
            def get_coords(id):
                lm = landmarks[id]
                return int(lm.x * w), int(lm.y * h)

            left_shoulder, right_shoulder, left_wrist, right_wrist, left_hip, right_hip, left_knee, right_knee, left_ankle, right_ankle = obtener_poses()

            # --- Acción: Levantar brazos ---
            promedio_hombros_y = (left_shoulder[1] + right_shoulder[1]) / 2
            promedio_munecas_y = (left_wrist[1] + right_wrist[1]) / 2
            if promedio_munecas_y < promedio_hombros_y:
                action = "levantar brazos"

            # --- Acción: Sentarse ---
            promedio_caderas_y = (left_hip[1] + right_hip[1]) / 2
            promedio_rodillas_y = (left_knee[1] + right_knee[1]) / 2
            if promedio_caderas_y > promedio_rodillas_y + 30:  # Umbral ajustable
                action = "sentarse"

            # --- Acción: Caminar ---
            posicion_actual_tobillos = (left_ankle[0], right_ankle[0])
            global posicion_previa_tobillos
            if posicion_previa_tobillos is not None:
                movimiento = abs(posicion_actual_tobillos[0] - posicion_previa_tobillos[0]) + \
                            abs(posicion_actual_tobillos[1] - posicion_previa_tobillos[1])
                if movimiento > umbral_movimiento:
                    action = "caminar"
            posicion_previa_tobillos = posicion_actual_tobillos

        except Exception as e:
            print(f"Error al procesar landmarks: {e}")

        # Dibujar landmarks
        mp_drawing.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS
        )

    # Mostrar acción en pantalla
    cv2.putText(frame, f"Accion: {action}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    print(f"[ACCION] {action}")

    # Mostrar ventana
    cv2.imshow('Reconocimiento de Acciones', frame)

    if cv2.waitKey(1) == 27:  # Presionar ESC para salir
        break

# Liberar recursos
cap.release()
cv2.destroyAllWindows()
pose.close()

[ACCION] ninguna
[ACCION] ninguna
[ACCION] caminar
[ACCION] caminar
[ACCION] caminar
[ACCION] caminar
[ACCION] caminar
[ACCION] ninguna
[ACCION] caminar
[ACCION] ninguna
[ACCION] caminar
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] caminar
[ACCION] caminar
[ACCION] caminar
[ACCION] ninguna
[ACCION] caminar
[ACCION] caminar
[ACCION] caminar
[ACCION] caminar
[ACCION] caminar
[ACCION] ninguna
[ACCION] caminar
[ACCION] caminar
[ACCION] ninguna
[ACCION] caminar
[ACCION] caminar
[ACCION] caminar
[ACCION] ninguna
[ACCION] caminar
[ACCION] caminar
[ACCION] caminar
[ACCION] caminar
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ninguna
[ACCION] ningu